# Toy generators

In [ ]:
import copy
import logging

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as sts

from numpy.typing import NDArray

import ppu
from ppu.generator import Circular, Moons, RingBlobs
from ppu.methods.mlp import MLP
from ppu.methods.tracin import get_random_resampled_tracin, get_random_tracin, get_loss_over_grid, get_proba_over_grid
from ppu.viz import plot_dense_binary_scatter, plot_dense_scatter, plot_ellipse_from_cov, plot_pdf_contours, GridPlot

## Config

In [ ]:
n_samples = 10_000
colors = ppu.viz.set_plot_style(True)
rng = np.random.Generator(np.random.PCG64DXSM(42))

## Definitions

In [ ]:
gen = Moons(class_sep=0.5)
# gen = Circular()
# gen = RingBlobs()

X_train, y_train = gen.rvs(10000)

# ax = plot_dense_binary_scatter(X_train, y_train)

### Single point

In [ ]:
import torch

In [ ]:
mlp = MLP(hidden_channels=[30, 100, 200, 200, 100, 50, 1], patience=40, frequency=3)
mlp.fit(X_train, y_train)

In [ ]:
grid = GridPlot(X=X_train, y=y_train, n_ticks=1000)

loss_0 = get_loss_over_grid(X=grid.X_grid, y=0, mlp=mlp)
loss_1 = get_loss_over_grid(X=grid.X_grid, y=1, mlp=mlp)

min_loss = np.minimum(loss_0, loss_1)
max_loss = np.maximum(loss_0, loss_1)

fig, axs = plt.subplots(figsize=(14, 8), ncols=2, nrows=2)
axs[0, 0].set_title("Maximum loss")
grid.plot(max_loss, overlay=True, ax=axs[0, 0])
axs[0, 1].set_title("Minimum loss")
grid.plot(min_loss, overlay=True, ax=axs[0, 1])
axs[1, 0].set_title("Loss class 0")
grid.plot(loss_0, overlay=True, ax=axs[1, 0])
axs[1, 1].set_title("Loss class 1")
grid.plot(loss_1, overlay=True, ax=axs[1, 1])

In [ ]:
from ppu.methods.point_tracin import TracIn

In [ ]:
mlp._opt_lr = 1e-5

In [ ]:
tracer = TracIn(mlp, X_train, y_train, n_points=5, n_ticks=2000, rng=rng, reset_optimizer=True)

In [ ]:
grid_l0, grid_l1, initial_loss_0, initial_loss_1 = tracer.trace_points(batch_size=500, patience=20, n_iters=100,)

In [ ]:
loss_sum = grid_l0 + grid_l1
max_loss = np.maximum(grid_l0, grid_l1)
min_loss = np.minimum(grid_l0, grid_l1)

initial_loss_sum = initial_loss_0 + initial_loss_1
initial_max_loss = np.maximum(initial_loss_0, initial_loss_1)
initial_min_loss = np.minimum(initial_loss_0, initial_loss_1)

tracin_0 = np.abs(grid_l0 - initial_loss_0)
tracin_1 = np.abs(grid_l1 - initial_loss_1)
sum_tracin = tracin_0 + tracin_1

max_tracin = np.maximum(tracin_0, tracin_1)
min_tracin = np.minimum(tracin_0, tracin_1)

min_max_tracin = np.abs(np.maximum(initial_loss_0, initial_loss_1) - np.minimum(grid_l0, grid_l1))
max_tracin_ratio = np.maximum((tracin_0 / (tracin_1 + 1)), (tracin_1 / (tracin_0 + 1)))

In [ ]:
center_kwargs = {
    "marker": ".",
    "s":10,
    "c":"orange",
}

fig, axs = plt.subplots(figsize=(18, 6), ncols=3)
ax = axs[0]
ax.set_title("Maximum TracIn ratio")
ax.hist(max_tracin_ratio.ravel(), bins="auto")

ax = axs[1]
ax.set_title("Maximum TracIn ratio")
tracer.plot(max_tracin_ratio, ax=ax)
ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

ax = axs[2]
ax.set_title("Maximum TracIn ratio")
tracer.plot(max_tracin_ratio, ax=ax, overlay=True)

In [ ]:
plot_centers = True
center_kwargs = {
    "marker": ".",
    "s":10,
    "c":"orange",
}

fig, axs = plt.subplots(figsize=(12, 12), ncols=2, nrows=3)
ax = axs[0, 0]
ax.set_title("Sum grid loss")
ax = tracer.plot(loss_sum, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[0, 1]
ax.set_title("Sum initial loss")
tracer.plot(initial_loss_sum, ax=ax)

ax = axs[1, 0]
ax.set_title("Max grid loss")
ax = tracer.plot(max_loss, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[1, 1]
ax.set_title("Max initial loss")
tracer.plot(initial_max_loss, ax=ax)

ax = axs[2, 0]
ax.set_title("Min grid loss")
ax = tracer.plot(min_loss, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[2, 1]
ax.set_title("Min initial loss")
tracer.plot(initial_min_loss, ax=ax)

In [ ]:
plot_centers = True
center_kwargs = {
    "marker": ".",
    "s":10,
    "c":"orange",
}

fig, axs = plt.subplots(figsize=(12, 12), ncols=2, nrows=3)
ax = axs[0, 0]
ax.set_title("TracIn - Label 0")
ax = tracer.plot(tracin_0, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[0, 1]
ax.set_title("TracIn - Label 1")
tracer.plot(tracin_1, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

ax = axs[1, 0]
ax.set_title("Sum TracIn")
ax = tracer.plot(sum_tracin, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[1, 1]
ax.set_title("Max TracIn")
tracer.plot(max_tracin, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

ax = axs[2, 0]
ax.set_title("Max Tracin ratio")
ax = tracer.plot(max_tracin_ratio, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[2, 1]
ax.set_title("Min-Max TracIn")
tracer.plot(min_max_tracin, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

In [ ]:
fig, axs = plt.subplots(figsize=(16, 4), ncols=4)
tracer.plot(tracin_0, ax=axs[0])
tracer.plot(tracin_1, ax=axs[1])
tracer.plot(tracin_0 + tracin_1, ax=axs[2])
tracer.plot(tracin_0 + tracin_1, ax=axs[3], overlay=True)

# 900 grid points

In [ ]:
mlp._opt_lr = 1e-5
tracer = TracIn(mlp, X_train, y_train, n_points=30, n_ticks=2000, rng=rng, reset_optimizer=True)

grid_l0, grid_l1, initial_loss_0, initial_loss_1 = tracer.trace_points(batch_size=500, patience=20, n_iters=100,)

loss_sum = grid_l0 + grid_l1
max_loss = np.maximum(grid_l0, grid_l1)
min_loss = np.minimum(grid_l0, grid_l1)

initial_loss_sum = initial_loss_0 + initial_loss_1
initial_max_loss = np.maximum(initial_loss_0, initial_loss_1)
initial_min_loss = np.minimum(initial_loss_0, initial_loss_1)

tracin_0 = np.abs(grid_l0 - initial_loss_0)
tracin_1 = np.abs(grid_l1 - initial_loss_1)
sum_tracin = tracin_0 + tracin_1

max_tracin = np.maximum(tracin_0, tracin_1)
min_tracin = np.minimum(tracin_0, tracin_1)

min_max_tracin = np.abs(np.maximum(initial_loss_0, initial_loss_1) - np.minimum(grid_l0, grid_l1))
max_tracin_ratio = np.maximum((tracin_0 / (tracin_1 + 1)), (tracin_1 / (tracin_0 + 1)))

center_kwargs = {
    "marker": ".",
    "s":10,
    "c":"orange",
}

fig, axs = plt.subplots(figsize=(18, 6), ncols=3)
ax = axs[0]
ax.set_title("Maximum TracIn ratio")
ax.hist(max_tracin_ratio.ravel(), bins="auto")

ax = axs[1]
ax.set_title("Maximum TracIn ratio")
tracer.plot(max_tracin_ratio, ax=ax)
#ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

ax = axs[2]
ax.set_title("Maximum TracIn ratio")
tracer.plot(max_tracin_ratio, ax=ax, overlay=True)

plot_centers = False

fig, axs = plt.subplots(figsize=(12, 12), ncols=2, nrows=3)
ax = axs[0, 0]
ax.set_title("TracIn - Label 0")
ax = tracer.plot(tracin_0, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[0, 1]
ax.set_title("TracIn - Label 1")
tracer.plot(tracin_1, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

ax = axs[1, 0]
ax.set_title("Sum TracIn")
ax = tracer.plot(sum_tracin, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[1, 1]
ax.set_title("Max TracIn")
tracer.plot(max_tracin, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

ax = axs[2, 0]
ax.set_title("Max Tracin ratio")
ax = tracer.plot(max_tracin_ratio, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[2, 1]
ax.set_title("Min-Max TracIn")
tracer.plot(min_max_tracin, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

plot_centers = False
center_kwargs = {
    "marker": ".",
    "s":10,
    "c":"orange",
}

fig, axs = plt.subplots(figsize=(12, 12), ncols=2, nrows=3)
ax = axs[0, 0]
ax.set_title("Sum grid loss")
ax = tracer.plot(loss_sum, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[0, 1]
ax.set_title("Sum initial loss")
tracer.plot(initial_loss_sum, ax=ax)

ax = axs[1, 0]
ax.set_title("Max grid loss")
ax = tracer.plot(max_loss, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[1, 1]
ax.set_title("Max initial loss")
tracer.plot(initial_max_loss, ax=ax)

ax = axs[2, 0]
ax.set_title("Min grid loss")
ax = tracer.plot(min_loss, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[2, 1]
ax.set_title("Min initial loss")
tracer.plot(initial_min_loss, ax=ax)

# 2500 points

In [ ]:
mlp._opt_lr = 1e-5
tracer = TracIn(mlp, X_train, y_train, n_points=50, n_ticks=2000, rng=rng, reset_optimizer=True)

grid_l0, grid_l1, initial_loss_0, initial_loss_1 = tracer.trace_points(batch_size=500, patience=20, n_iters=100,)

loss_sum = grid_l0 + grid_l1
max_loss = np.maximum(grid_l0, grid_l1)
min_loss = np.minimum(grid_l0, grid_l1)

initial_loss_sum = initial_loss_0 + initial_loss_1
initial_max_loss = np.maximum(initial_loss_0, initial_loss_1)
initial_min_loss = np.minimum(initial_loss_0, initial_loss_1)

tracin_0 = np.abs(grid_l0 - initial_loss_0)
tracin_1 = np.abs(grid_l1 - initial_loss_1)
sum_tracin = tracin_0 + tracin_1

max_tracin = np.maximum(tracin_0, tracin_1)
min_tracin = np.minimum(tracin_0, tracin_1)

min_max_tracin = np.abs(np.maximum(initial_loss_0, initial_loss_1) - np.minimum(grid_l0, grid_l1))
max_tracin_ratio = np.maximum((tracin_0 / (tracin_1 + 1)), (tracin_1 / (tracin_0 + 1)))

center_kwargs = {
    "marker": ".",
    "s":10,
    "c":"orange",
}

fig, axs = plt.subplots(figsize=(18, 6), ncols=3)
ax = axs[0]
ax.set_title("Maximum TracIn ratio")
ax.hist(max_tracin_ratio.ravel(), bins="auto")

ax = axs[1]
ax.set_title("Maximum TracIn ratio")
tracer.plot(max_tracin_ratio, ax=ax)
#ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

ax = axs[2]
ax.set_title("Maximum TracIn ratio")
tracer.plot(max_tracin_ratio, ax=ax, overlay=True)

plot_centers = False

fig, axs = plt.subplots(figsize=(12, 12), ncols=2, nrows=3)
ax = axs[0, 0]
ax.set_title("TracIn - Label 0")
ax = tracer.plot(tracin_0, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[0, 1]
ax.set_title("TracIn - Label 1")
tracer.plot(tracin_1, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

ax = axs[1, 0]
ax.set_title("Sum TracIn")
ax = tracer.plot(sum_tracin, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[1, 1]
ax.set_title("Max TracIn")
tracer.plot(max_tracin, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

ax = axs[2, 0]
ax.set_title("Max Tracin ratio")
ax = tracer.plot(max_tracin_ratio, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[2, 1]
ax.set_title("Min-Max TracIn")
tracer.plot(min_max_tracin, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

plot_centers = False
center_kwargs = {
    "marker": ".",
    "s":10,
    "c":"orange",
}

fig, axs = plt.subplots(figsize=(12, 12), ncols=2, nrows=3)
ax = axs[0, 0]
ax.set_title("Sum grid loss")
ax = tracer.plot(loss_sum, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[0, 1]
ax.set_title("Sum initial loss")
tracer.plot(initial_loss_sum, ax=ax)

ax = axs[1, 0]
ax.set_title("Max grid loss")
ax = tracer.plot(max_loss, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[1, 1]
ax.set_title("Max initial loss")
tracer.plot(initial_max_loss, ax=ax)

ax = axs[2, 0]
ax.set_title("Min grid loss")
ax = tracer.plot(min_loss, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[2, 1]
ax.set_title("Min initial loss")
tracer.plot(initial_min_loss, ax=ax)

# 5625 points

In [ ]:
mlp._opt_lr = 1e-5
tracer = TracIn(mlp, X_train, y_train, n_points=75, n_ticks=2000, rng=rng, reset_optimizer=True)

grid_l0, grid_l1, initial_loss_0, initial_loss_1 = tracer.trace_points(batch_size=500, patience=20, n_iters=100,)

In [ ]:
loss_sum = grid_l0 + grid_l1
max_loss = np.maximum(grid_l0, grid_l1)
min_loss = np.minimum(grid_l0, grid_l1)

initial_loss_sum = initial_loss_0 + initial_loss_1
initial_max_loss = np.maximum(initial_loss_0, initial_loss_1)
initial_min_loss = np.minimum(initial_loss_0, initial_loss_1)

tracin_0 = np.abs(grid_l0 - initial_loss_0)
tracin_1 = np.abs(grid_l1 - initial_loss_1)
sum_tracin = tracin_0 + tracin_1

max_tracin = np.maximum(tracin_0, tracin_1)
min_tracin = np.minimum(tracin_0, tracin_1)

min_max_tracin = np.abs(np.maximum(initial_loss_0, initial_loss_1) - np.minimum(grid_l0, grid_l1))
max_tracin_ratio = np.maximum((tracin_0 / (tracin_1 + 1)), (tracin_1 / (tracin_0 + 1)))

In [ ]:
center_kwargs = {
    "marker": ".",
    "s":10,
    "c":"orange",
}

fig, axs = plt.subplots(figsize=(18, 6), ncols=3)
ax = axs[0]
ax.set_title("Maximum TracIn ratio")
ax.hist(max_tracin_ratio.ravel(), bins="auto")

ax = axs[1]
ax.set_title("Maximum TracIn ratio")
tracer.plot(max_tracin_ratio, ax=ax)
#ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

ax = axs[2]
ax.set_title("Maximum TracIn ratio")
tracer.plot(max_tracin_ratio, ax=ax, overlay=True)

In [ ]:
plot_centers = False

fig, axs = plt.subplots(figsize=(12, 12), ncols=2, nrows=3)
ax = axs[0, 0]
ax.set_title("TracIn - Label 0")
ax = tracer.plot(tracin_0, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[0, 1]
ax.set_title("TracIn - Label 1")
tracer.plot(tracin_1, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

ax = axs[1, 0]
ax.set_title("Sum TracIn")
ax = tracer.plot(sum_tracin, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[1, 1]
ax.set_title("Max TracIn")
tracer.plot(max_tracin, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

ax = axs[2, 0]
ax.set_title("Max Tracin ratio")
ax = tracer.plot(max_tracin_ratio, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[2, 1]
ax.set_title("Min-Max TracIn")
tracer.plot(min_max_tracin, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

In [ ]:
fig, axs = plt.subplots(figsize=(12, 12), ncols=2, nrows=3)
ax = axs[0, 0]
ax.set_title("Sum grid loss")
ax = tracer.plot(loss_sum, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[0, 1]
ax.set_title("Sum initial loss")
tracer.plot(initial_loss_sum, ax=ax)

ax = axs[1, 0]
ax.set_title("Max grid loss")
ax = tracer.plot(max_loss, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[1, 1]
ax.set_title("Max initial loss")
tracer.plot(initial_max_loss, ax=ax)

ax = axs[2, 0]
ax.set_title("Min grid loss")
ax = tracer.plot(min_loss, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[2, 1]
ax.set_title("Min initial loss")
tracer.plot(initial_min_loss, ax=ax)

# Hole

In [ ]:
def create_hole_mask(arr, lb, ub):
    x0 = (arr[:, 0] > lb) & (arr[:, 0]  < ub)
    x1 = (arr[:, 1] > lb) & (arr[:, 1]  < ub)
    return x0 & x1

In [ ]:
gen = Circular(class_sep=1.3)

In [ ]:
X_train, y_train = gen.rvs(10000)

ax = plot_dense_binary_scatter(X_train, y_train)

In [ ]:
n_extra = 100
label = 1
centroid = (0, 0)
scale = 0.02

x_train_mask = create_hole_mask(X_train, -0.1, 0.1)

X_hole = X_train[~x_train_mask, :].copy()
y_hole = y_train[~x_train_mask].copy()

ax = plot_dense_binary_scatter(X_hole, y_hole)

In [ ]:
mlp = MLP(hidden_channels=[30, 100, 200, 200, 100, 50, 1], patience=40, frequency=3)
mlp.fit(X_hole, y_hole)

In [ ]:
grid = GridPlot(X=X_hole, y=y_hole, n_ticks=1000)

loss_0 = get_loss_over_grid(X=grid.X_grid, y=0, mlp=mlp)
loss_1 = get_loss_over_grid(X=grid.X_grid, y=1, mlp=mlp)

min_loss = np.minimum(loss_0, loss_1)
max_loss = np.maximum(loss_0, loss_1)

fig, axs = plt.subplots(figsize=(14, 8), ncols=2, nrows=2)
axs[0, 0].set_title("Maximum loss")
grid.plot(max_loss, overlay=True, ax=axs[0, 0])
axs[0, 1].set_title("Minimum loss")
grid.plot(min_loss, overlay=True, ax=axs[0, 1])
axs[1, 0].set_title("Loss class 0")
grid.plot(loss_0, overlay=True, ax=axs[1, 0])
axs[1, 1].set_title("Loss class 1")
grid.plot(loss_1, overlay=True, ax=axs[1, 1])

In [ ]:
mlp._opt_lr = 1e-5
tracer = TracIn(mlp, X_hole, y_hole, n_points=50, n_ticks=2000, rng=rng, reset_optimizer=True)

In [ ]:
tracer._create_grids(X_hole, n_points=50, n_ticks=2000, border=1)

In [ ]:
#ax = tracer.plot(max_tracin_ratio, overlay=True)
fig, ax = plt.subplots(figsize=(8, 8))
ax = plot_dense_binary_scatter(X_hole, y_hole, ax=ax)
ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

In [ ]:
grid_l0, grid_l1, initial_loss_0, initial_loss_1 = tracer.trace_points(batch_size=500, patience=30, n_iters=100,)

In [ ]:
loss_sum = grid_l0 + grid_l1
max_loss = np.maximum(grid_l0, grid_l1)
min_loss = np.minimum(grid_l0, grid_l1)

initial_loss_sum = initial_loss_0 + initial_loss_1
initial_max_loss = np.maximum(initial_loss_0, initial_loss_1)
initial_min_loss = np.minimum(initial_loss_0, initial_loss_1)

tracin_0 = np.abs(grid_l0 - initial_loss_0)
tracin_1 = np.abs(grid_l1 - initial_loss_1)
sum_tracin = tracin_0 + tracin_1

max_tracin = np.maximum(tracin_0, tracin_1)
min_tracin = np.minimum(tracin_0, tracin_1)

min_max_tracin = np.abs(np.maximum(initial_loss_0, initial_loss_1) - np.minimum(grid_l0, grid_l1))
max_tracin_ratio = np.maximum((tracin_0 / (tracin_1 + 1)), (tracin_1 / (tracin_0 + 1)))

In [ ]:
center_kwargs = {
    "marker": ".",
    "s":10,
    "c":"orange",
}

fig, axs = plt.subplots(figsize=(18, 6), ncols=3)
ax = axs[0]
ax.set_title("Maximum TracIn ratio")
ax.hist(max_tracin_ratio.ravel(), bins="auto")

ax = axs[1]
ax.set_title("Maximum TracIn ratio")
tracer.plot(max_tracin_ratio, ax=ax)
#ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

ax = axs[2]
ax.set_title("Maximum TracIn ratio")
tracer.plot(max_tracin_ratio, ax=ax, overlay=True)

In [ ]:
plot_centers = False

fig, axs = plt.subplots(figsize=(12, 12), ncols=2, nrows=3)
ax = axs[0, 0]
ax.set_title("TracIn - Label 0")
ax = tracer.plot(tracin_0, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[0, 1]
ax.set_title("TracIn - Label 1")
tracer.plot(tracin_1, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

ax = axs[1, 0]
ax.set_title("Sum TracIn")
ax = tracer.plot(sum_tracin, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[1, 1]
ax.set_title("Max TracIn")
tracer.plot(max_tracin, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

ax = axs[2, 0]
ax.set_title("Max Tracin ratio")
ax = tracer.plot(max_tracin_ratio, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[2, 1]
ax.set_title("Min-Max TracIn")
tracer.plot(min_max_tracin, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

In [ ]:
plot_centers = False

fig, axs = plt.subplots(figsize=(12, 12), ncols=2, nrows=3)
ax = axs[0, 0]
ax.set_title("Sum grid loss")
ax = tracer.plot(loss_sum, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[0, 1]
ax.set_title("Sum initial loss")
tracer.plot(initial_loss_sum, ax=ax)

ax = axs[1, 0]
ax.set_title("Max grid loss")
ax = tracer.plot(max_loss, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[1, 1]
ax.set_title("Max initial loss")
tracer.plot(initial_max_loss, ax=ax)

ax = axs[2, 0]
ax.set_title("Min grid loss")
ax = tracer.plot(min_loss, ax=ax)
if plot_centers:
    ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)
ax = axs[2, 1]
ax.set_title("Min initial loss")
tracer.plot(initial_min_loss, ax=ax)

## Point grid-trace

In [ ]:
mlp._opt_lr = 5e-3
tracer = TracIn(mlp, X_hole, y_hole, n_points=2, n_ticks=1000, rng=rng, reset_optimizer=True)

In [ ]:
n_iters = 100

grid_loss_l0, grid_loss_l1, l0_loss_store, l1_loss_store = tracer.trace_point((0., 0.), batch_size=500, n_iters=n_iters, weight=250.0)

In [ ]:
tracin_l1 = grid_loss_l1[0, :] - grid_loss_l1[1:, :]
tracin_l0 = grid_loss_l0[0, :] - grid_loss_l0[1:, :]

tracin = tracin_l0 + tracin_l1

In [ ]:
fig, axs = plt.subplots(figsize=(10, 8), ncols=2)
ax = axs[0]
ax.plot(l1_loss_store[:, -1], label="point lost")
ax.plot(l1_loss_store.mean(1), label="mean batch loss")
ax.plot(l1_loss_store[:, :-1].mean(1), label="mean sample loss")
ax.legend()
ax = axs[1]
ax.plot(l1_loss_store[:, -1], label="point lost")
ax.plot(l1_loss_store.mean(1), label="mean batch loss")
ax.plot(l1_loss_store[:, :-1].mean(1), label="mean sample loss")
ax.legend()
ax.set_yscale("log")
#ax.set_yscale("log")

In [ ]:
plt.plot(np.diff(l1_loss_store[0, -1] - l1_loss_store[1:, -1]))

In [ ]:
plt.plot(np.diff(l0_loss_store[0, -1] - l0_loss_store[1:, -1]))

In [ ]:
fig, axs = plt.subplots(figsize=(15, 12), ncols=5, nrows=4)
axs_ = axs.ravel()
for i, ii in enumerate([i for i in range(1, n_iters + 1) if i % 5 == 0]):
    axs_[i].set_title(f"iter: {ii}")
    tracer.plot(grid_loss_l1[ii, :], ax=axs_[i], vmin=0.0)
fig.suptitle("Grid loss | y = 1", fontsize=18)
fig.tight_layout()
# fig.savefig("max_loss_moons_single_point_overlay.png", dpi=300)

In [ ]:
fig, axs = plt.subplots(figsize=(15, 12), ncols=5, nrows=4)
axs_ = axs.ravel()
for i, ii in enumerate([i for i in range(1, n_iters + 1) if i % 5 == 0]):
    axs_[i].set_title(f"iter: {ii}")
    tracer.plot(tracin_l1[ii - 1, :], ax=axs_[i], vmin=0.0)
fig.suptitle("TraIn | y = 1", fontsize=18)
fig.tight_layout()

In [ ]:
fig, axs = plt.subplots(figsize=(15, 12), ncols=5, nrows=4)
axs_ = axs.ravel()
for i, ii in enumerate([i for i in range(1, n_iters + 1) if i % 5 == 0]):
    axs_[i].set_title(f"iter: {ii}")
    tracer.plot(tracin_l0[ii - 1, :], ax=axs_[i], vmin=0.0)
fig.suptitle("TraIn | y = 0", fontsize=18)
fig.tight_layout()

In [ ]:
fig, axs = plt.subplots(figsize=(15, 12), ncols=5, nrows=4)
axs_ = axs.ravel()
for i, ii in enumerate([i for i in range(1, n_iters + 1) if i % 5 == 0]):
    axs_[i].set_title(f"iter: {ii}")
    tracer.plot(tracin[ii - 1, :], ax=axs_[i], vmin=0)
fig.suptitle("TracIn", fontsize=18)
fig.tight_layout()

In [ ]:
expected_tracin = np.clip(tracin[59, :], a_min=0, a_max=np.inf)

In [ ]:
fig, axs = plt.subplots(figsize=(18, 6), ncols=3)
ax = axs[0]
ax.set_title("Expected TracIn")
ax.hist(expected_tracin.ravel(), bins="auto")

ax = axs[1]
ax.set_title("Expected TracIn")
tracer.plot(expected_tracin, ax=ax, vmin=0.0)
#ax.scatter(tracer.point_X_grid[:, 0], tracer.point_X_grid[:, 1], **center_kwargs)

ax = axs[2]
ax.set_title("Expected TracIn")
tracer.plot(expected_tracin, ax=ax, overlay=True, vmin=0.0)

### Counter example

In [ ]:
X_train, y_train = gen.rvs(10000)

In [ ]:
mlp = MLP(hidden_channels=[30, 100, 200, 200, 100, 50, 1], patience=40, frequency=3)
mlp.fit(X_train, y_train)

In [ ]:
mlp._opt_lr = 5e-3
tracer = TracIn(mlp, X_train, y_train, n_points=2, n_ticks=1000, rng=rng, reset_optimizer=True)

In [ ]:
n_iters = 100

grid_loss_l0, grid_loss_l1, l0_loss_store, l1_loss_store = tracer.trace_point((0., 0.), batch_size=500, n_iters=n_iters, weight=250.0)

In [ ]:
tracin_l1 = grid_loss_l1[0, :] - grid_loss_l1[1:, :]
tracin_l0 = grid_loss_l0[0, :] - grid_loss_l0[1:, :]

tracin = tracin_l0 + tracin_l1

In [ ]:
fig, axs = plt.subplots(figsize=(10, 8), ncols=2)
ax = axs[0]
ax.plot(l1_loss_store[:, -1], label="point lost")
ax.plot(l1_loss_store.mean(1), label="mean batch loss")
ax.plot(l1_loss_store[:, :-1].mean(1), label="mean sample loss")
ax.legend()
ax = axs[1]
ax.plot(l1_loss_store[:, -1], label="point lost")
ax.plot(l1_loss_store.mean(1), label="mean batch loss")
ax.plot(l1_loss_store[:, :-1].mean(1), label="mean sample loss")
ax.legend()
ax.set_yscale("log")
#ax.set_yscale("log")

In [ ]:
plt.plot(l1_loss_store[0, -1] - l1_loss_store[1:, -1])

In [ ]:
plt.plot(np.diff(l1_loss_store[0, -1] - l1_loss_store[1:, -1]))

In [ ]:
plt.plot(np.diff(l0_loss_store[0, -1] - l0_loss_store[1:, -1]))

In [ ]:
fig, axs = plt.subplots(figsize=(15, 12), ncols=5, nrows=4)
axs_ = axs.ravel()
for i, ii in enumerate([i for i in range(1, n_iters + 1) if i % 5 == 0]):
    axs_[i].set_title(f"iter: {ii}")
    tracer.plot(grid_loss_l1[ii, :], ax=axs_[i], vmin=0.0)
fig.suptitle("Grid loss | y = 1", fontsize=18)
fig.tight_layout()
# fig.savefig("max_loss_moons_single_point_overlay.png", dpi=300)

In [ ]:
fig, axs = plt.subplots(figsize=(15, 12), ncols=5, nrows=4)
axs_ = axs.ravel()
for i, ii in enumerate([i for i in range(1, n_iters + 1) if i % 5 == 0]):
    axs_[i].set_title(f"iter: {ii}")
    tracer.plot(tracin_l0[ii - 1, :], ax=axs_[i], vmin=0.0)
fig.suptitle("TraIn | y = 0", fontsize=18)
fig.tight_layout()

In [ ]:
fig, axs = plt.subplots(figsize=(15, 12), ncols=5, nrows=4)
axs_ = axs.ravel()
for i, ii in enumerate([i for i in range(1, n_iters + 1) if i % 5 == 0]):
    axs_[i].set_title(f"iter: {ii}")
    tracer.plot(tracin_l1[ii - 1, :], ax=axs_[i], vmin=0.0)
fig.suptitle("TraIn | y = 1", fontsize=18)
fig.tight_layout()

In [ ]:
fig, axs = plt.subplots(figsize=(15, 12), ncols=5, nrows=4)
axs_ = axs.ravel()
for i, ii in enumerate([i for i in range(1, n_iters + 1) if i % 5 == 0]):
    axs_[i].set_title(f"iter: {ii}")
    tracer.plot(tracin[ii - 1, :], ax=axs_[i], vmin=0)
fig.suptitle("TracIn", fontsize=18)
fig.tight_layout()

## Conclusions

We should fit each point in a batch until the tracin score of that point has converged.

* The learning rate may have to higher than normal
* weighting of the point of interest may be required.

How to we balance setting the weight? Too high of weight and the model will adapt even if there are sufficiently many points close by